# Models

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import json

In [ ]:
data = pd.read_csv('dataset_clean.csv', delimiter=',', dtype={
    'PRIORITY': 'category',
    'COMPONENT': 'category',
    'LABEL': 'category',
    'URGENCY': 'category',
    'IMPACT': 'category',
    'ISSUE_CATEGORY': 'category',
    'ASSIGNEE': 'category',
    'POZISYON': 'category',
    'IS_BILGISI': 'category',
    'Kademe': 'category',
    'Complition Time': 'float32',
    'Total_Assignee': 'int8',
    'Total_Worklog_Assginee': 'int8',
    'Total_Log_Hours_Assignee': 'float32',
    'COMMENTOR_COUNT': 'int8',
    'COMMENT_COUNT': 'int8',
    'WAITING_HOURS_AT_CUSTOMER': 'float32',
    'ISSUE_TYPE': 'category',
})
data['URGENCY'] = data['URGENCY'].map({'Low': 0, 'Medium': 1, 'High': 2})
X = data.drop(['URGENCY'], axis=1)
X = pd.get_dummies(X)
y = data['URGENCY']
selected_features = [
    ['IMPACT_Operation Continue With Workaround', 'Complition Time',
       'PRIORITY_Major', 'WAITING_HOURS_AT_CUSTOMER',
       'IMPACT_Some Processes Stop on Production'],
    ['WAITING_HOURS_AT_CUSTOMER',
       'IMPACT_Operation Continue With Workaround', 'PRIORITY_Minor',
       'PRIORITY_Major', 'Complition Time'],
    ['COMMENT_COUNT', 'Complition Time',
       'PRIORITY_Major', 'ISSUE_TYPE_Incident',
       'IMPACT_Some Processes Stop on Production']
]
val_dict = {}

In [ ]:
def take_average(res: list, algorithm_name: str):
    avg_accuracy, avg_precision, avg_recall, avg_f1 = 0, 0, 0, 0
    avg_f1_micro, avg_f1_macro = 0, 0
    for acc, prec, rec, f1, f1_micro, f1_macro in res:
        avg_accuracy += acc
        avg_precision += prec
        avg_recall += rec
        avg_f1 += f1
        avg_f1_micro += f1_micro
        avg_f1_macro += f1_macro

    avg_accuracy = avg_accuracy / len(res)
    avg_precision = avg_precision / len(res)
    avg_recall = avg_recall / len(res)
    avg_f1 = avg_f1 / len(res)
    avg_f1_micro = avg_f1_micro / len(res)
    avg_f1_macro = avg_f1_macro / len(res)
    
    print(f"Average Metrics for {algorithm_name}")
    print(f"Accuracy {avg_accuracy}")
    print(f"Precision {avg_precision}")
    print(f"Recall {avg_recall}")
    print(f"F1 {avg_f1}\n")
    print(f"F1-micro {avg_f1_micro}\n")
    print(f"F1-macro {avg_f1_macro}\n")

    val_dict[algorithm_name] = [avg_accuracy, avg_precision, avg_recall, avg_f1, avg_f1_micro, avg_f1_macro]

def print_confusion_matrix(metrics_cv: list, all_y_tests: list, all_y_preds: list, algorithm_name: str):
    # finds best result based on f1 metric
    f1_index = 3
    f1_results = list(map(lambda m: m[f1_index], metrics_cv))
    best_metric_index = f1_results.index(max(f1_results))
    
    y_test = all_y_tests[best_metric_index]
    y_pred = all_y_preds[best_metric_index]

    print(f"Best Metric based on F1 for {algorithm_name}")
    print(f"Accuracy {metrics_cv[best_metric_index][0]}")
    print(f"Precision {metrics_cv[best_metric_index][1]}")
    print(f"Recall {metrics_cv[best_metric_index][2]}")
    print(f"F1 {metrics_cv[best_metric_index][3]}\n")
    print(f"F1-micro {metrics_cv[best_metric_index][4]}\n")
    print(f"F1-macro {metrics_cv[best_metric_index][5]}\n")

    print(f"Confusion Matrix {algorithm_name}")
    print(confusion_matrix(y_test, y_pred))


## SVM

In [ ]:
def svm(X, y):
    svm = SVC()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42    )
    metrics_cv = []
    all_y_tests = []
    all_y_preds = []
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        svm.fit(X_train, y_train)
        y_pred = svm.predict(X_test)


        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)


        all_y_tests.append(y_test)
        all_y_preds.append(y_pred)

        metrics_cv.append([accuracy, precision, recall, f1, f1_micro, f1_macro])
    return metrics_cv, all_y_tests, all_y_preds

## Stochastic Gradient Descent

In [ ]:
from sklearn.linear_model import SGDClassifier

def sgd(X, y):
    sgd = SGDClassifier()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    metrics_cv = []

    all_y_tests = []
    all_y_preds = []

    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        sgd.fit(X_train, y_train)
        y_pred = sgd.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

        metrics_cv.append([accuracy, precision, recall, f1, f1_micro, f1_macro])
        all_y_tests.append(y_test)
        all_y_preds.append(y_pred)

    return metrics_cv, all_y_tests, all_y_preds

## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def decision_tree(X, y):
    dt = DecisionTreeClassifier()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    metrics_cv = []
    all_y_preds = []
    all_y_tests = []

    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        dt.fit(X_train, y_train)
        y_pred = dt.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

        metrics_cv.append([accuracy, precision, recall, f1, f1_micro, f1_macro])
        all_y_tests.append(y_test)
        all_y_preds.append(y_pred)

    return metrics_cv, all_y_tests, all_y_preds

## Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

def naive_bayes(X, y):
    nb = GaussianNB()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    metrics_cv = []

    all_y_preds = []
    all_y_tests = []


    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        nb.fit(X_train, y_train)
        y_pred = nb.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

        metrics_cv.append([accuracy, precision, recall, f1, f1_micro, f1_macro])
        all_y_preds.append(y_pred)
        all_y_tests.append(y_test)

    return metrics_cv, all_y_tests, all_y_preds

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def random_forest(X, y):
    nn = RandomForestClassifier()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    metrics_cv = []

    all_y_preds = []
    all_y_tests = []


    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        nn.fit(X_train, y_train)
        y_pred = nn.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

        metrics_cv.append([accuracy, precision, recall, f1, f1_micro, f1_macro])

        all_y_preds.append(y_pred)
        all_y_tests.append(y_test)

    return metrics_cv, all_y_tests, all_y_preds

## Multi-Layer Perceptron

In [ ]:
from sklearn.neural_network import MLPClassifier 

def mlp(X, y):
    nn = MLPClassifier(hidden_layer_sizes=(5, 2), max_iter=500)

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    metrics_cv = []

    all_y_preds = []
    all_y_tests = []

    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        nn.fit(X_train, y_train)
        y_pred = nn.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

        metrics_cv.append([accuracy, precision, recall, f1, f1_micro, f1_macro])

        all_y_preds.append(y_pred)
        all_y_tests.append(y_test)

    return metrics_cv, all_y_tests, all_y_preds

## Experiment

In [ ]:
for i in range(len(selected_features)):
    X_selected = X[selected_features[i]]
    print(f"Feature Set #{i}")
    print(selected_features[i])
    results = [svm(X_selected, y), sgd(X_selected, y), decision_tree(X_selected, y), naive_bayes(X_selected, y), random_forest(X_selected, y), mlp(X_selected, y)]
    names = ["Support Vector Machine", "Stochastic Gradient Descent", "Decision Tree", "Naive Bayes", "Random Forest", "Multi-Layer Perceptron"]
    j = 0
    np.save(f"results_f{i}.npy", [row[0] for row in results])
   # with open('results.json', 'w') as f:
     # json.dump([result.tolist() for result in results], f)
    for metrics_cv, all_y_tests, all_y_preds in results:
        neural_network_result = take_average(metrics_cv, names[j])
        print_confusion_matrix(metrics_cv, all_y_tests, all_y_preds, names[j])
        j += 1

In [ ]:
from scipy.stats import ttest_ind
results = np.load("results_f0.npy", allow_pickle=True)
d = dict(enumerate(results.flatten(), 1))
print(results)

# names = ["Support Vector Machine", "Stochastic Gradient Descent", "Decision Tree", "Naive Bayes", "Random Forest", "Multi-Layer Perceptron"]
# best models: 4 and 2
# Perform the t-test
t_statistic, p_value = ttest_ind(results[4].flatten(), results[2].flatten())

# Display the results
print("t-statistic:", t_statistic)
print("p-value:", p_value)

def f1_bar_chart_best(results):
    algorithms = ["Support Vector Machine", "Stochastic Gradient Descent", "Decision Tree", "Naive Bayes", "Random Forest", "Multi-Layer Perceptron"]
    values = [value[3][3] for value in results]  # Extract the fourth value from each key-value pair
    print(values)

    num_bars = len(results)

    # Create an array of indices for the bars
    indices = np.arange(num_bars)

    # Set the width of each bar
    bar_width = 0.5

    fig, ax = plt.subplots(figsize=(10, 8))  # Set the figure size

    # Generate a list of colors using the 'tab10' colormap
    colors = plt.cm.tab10(np.arange(num_bars))

    # Plot the bars with tab10 colors
    ax.bar(indices, values, width=bar_width, color=colors)

    # Set the x-axis ticks and labels
    ax.set_xticks(indices)
    ax.set_xticklabels(algorithms, rotation='vertical')  # Rotate x tick labels vertically

    # Set the y-axis label
    ax.set_ylabel('Values')

    # Set the plot title
    ax.set_title('F1 Scores (Feature Set I)')

    # Show the plot
    plt.show()

f1_bar_chart_best(results)

def plot_chart_best(data_dict):
    algorithms = ["Support Vector Machine", "Stochastic Gradient Descent", "Decision Tree", "Naive Bayes", "Random Forest", "Multi-Layer Perceptron"]
    values = [value[3] for value in results]  # Extract the fourth value from each key-value pair
    #values = list(data_dict.values())

    legend_names = ['Accuracy', 'Precision', 'Recall', 'F1']

    num_bars = len(algorithms)
    num_values = len(values[0])

    # Create an array of indices for the bars
    indices = np.arange(num_bars)

    # Set the width of each bar
    bar_width = 0.07

    # Set the spacing between bars
    spacing = 0.05

    fig, ax = plt.subplots(figsize=(10, 8))

    for i in range(num_values):
        # Calculate the x-coordinate for each group of bars
        x = indices + i * (bar_width + spacing)

        # Extract the values for the current iteration
        y = [value[i] for value in values]

        # Plot the bars
        ax.bar(x, y, width=bar_width)

    # Set the x-axis ticks and labels
    ax.set_xticks(indices + (bar_width * num_values + spacing * (num_values - 1)) / 2)
    ax.set_xticklabels(algorithms, rotation='vertical')

    # Set the y-axis label
    ax.set_ylabel('Values')

    # Set the plot title
    ax.set_title('All Metrics per Model')

    # Display the legend
    ax.legend(legend_names)

    # Show the plot
    plt.show()

plot_chart_best(results)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_bar_chart(data_dict):
    algorithms = list(data_dict.keys())
    values = list(data_dict.values())

    legend_names = ['Accuracy', 'Precision', 'Recall', 'F1']

    num_bars = len(algorithms)
    num_values = len(values[0])

    # Create an array of indices for the bars
    indices = np.arange(num_bars)

    # Set the width of each bar
    bar_width = 0.07

    # Set the spacing between bars
    spacing = 0.05

    fig, ax = plt.subplots(figsize=(10, 8))

    for i in range(num_values):
        # Calculate the x-coordinate for each group of bars
        x = indices + i * (bar_width + spacing)

        # Extract the values for the current iteration
        y = [value[i] for value in values]

        # Plot the bars
        ax.bar(x, y, width=bar_width)

    # Set the x-axis ticks and labels
    ax.set_xticks(indices + (bar_width * num_values + spacing * (num_values - 1)) / 2)
    ax.set_xticklabels(algorithms, rotation='vertical')

    # Set the y-axis label
    ax.set_ylabel('Values')

    # Set the plot title
    ax.set_title('All Metrics per Model')

    # Display the legend
    ax.legend(legend_names)

    # Show the plot
    plt.show()

def f1_bar_chart(data_dict):
    algorithms = list(data_dict.keys())
    values = [value[3] for value in data_dict.values()]  # Extract the fourth value from each key-value pair

    num_bars = len(algorithms)

    # Create an array of indices for the bars
    indices = np.arange(num_bars)

    # Set the width of each bar
    bar_width = 0.5

    fig, ax = plt.subplots(figsize=(10, 8))  # Set the figure size

    # Generate a list of colors using the 'tab10' colormap
    colors = plt.cm.tab10(np.arange(num_bars))

    # Plot the bars with tab10 colors
    ax.bar(indices, values, width=bar_width, color=colors)

    # Set the x-axis ticks and labels
    ax.set_xticks(indices)
    ax.set_xticklabels(algorithms, rotation='vertical')  # Rotate x tick labels vertically

    # Set the y-axis label
    ax.set_ylabel('Values')

    # Set the plot title
    ax.set_title('F1 scores Bar Plot')

    # Show the plot
    plt.show()

In [ ]:
plot_bar_chart(val_dict)

In [ ]:
f1_bar_chart(val_dict)